# 0. Controls-96 on Kaggle: Platform Exception Only
Active study: P, B3 (seed 44 only), C1, C2, B4, BM x seeds 42/43/44 (B3 only seed 44), 20 epochs each, no early stop or extension -> **16 jobs**. B1/B2 are disabled. BM3D 200 cubed is resized to 96 and trained with the full 3-branch CrossGate model. Single P100 runs one process; dual T4 runs two independent jobs, one GPU each, never DDP or pooled 32 GB.
Default session subset is at most **2 runs**, with a **10-hour budget** including checkpoint grace. These are scheduling limits, not a promise that either run will finish. Resume unfinished jobs in later sessions; no extra epochs. Measure epoch time and VRAM on Kaggle before committing the full budget.
Enable Kaggle **Internet**, GPU, and Secrets `HF_TOKEN` / `WANDB_API_KEY`. W&B + Kaggle outputs are the explicit exception to Drive: no Drive API, credentials, or mount. Interactive outputs are ephemeral: **Save Version or download best weights before ending the session**. See `docs/controls-96-kaggle.md`.
CPU synthetic smoke: set `CTRL_KAGGLE_SMOKE=1` before setup. Runs all six active specifications for one epoch/seed (B3 on seed 44), offline W&B, no downloads or real backbone training. Real-run defaults remain enabled in this notebook.

## 1. Setup: Isolated Hardware Check Before Imports or Data
Reference: BM3D Kaggle P100 wheel profile, with protected dependency resolution instead of --no-deps. Working P100/T4 stacks are retained. Only incompatible P100 kernels may auto-repair to Torch 2.5.1 + torchvision 0.20.1 + torchaudio 2.5.1 cu121 (Python 3.9-3.12). The parent never imports Torch. When this kernel never imported Torch, a successful repair continues in the same session (Save-Version safe) after a fresh re-probe; if Torch was already imported, or the re-probe still lacks sm_60, setup stops and requests a restart. No model objects are reloaded. Setup never trains. Attach the intended checkout; an existing clone is not pulled automatically.

In [ ]:
import os
SMOKE = os.environ.get('CTRL_KAGGLE_SMOKE', '0') == '1'
if not SMOKE:
    os.environ['CTRL_KAGGLE_SETUP_PENDING'] = '1'
import sys, subprocess, importlib.util, tempfile
from pathlib import Path
# ===== BOOTSTRAP POLICY (edit repair opt-in only; no study changes) =====
AUTO_REPAIR_P100 = os.environ.get('CTRL_KAGGLE_AUTO_REPAIR_P100', '1') == '1'
if SMOKE:
    os.environ['CUDA_VISIBLE_DEVICES'] = ''
    os.environ.setdefault('CTRL_KAGGLE_TEMP_ROOT', tempfile.mkdtemp(prefix='controls-cache-'))
if os.environ.get('CTRL_KAGGLE_REPO_ROOT'):
    REPO = Path(os.environ['CTRL_KAGGLE_REPO_ROOT']).resolve()
elif SMOKE:
    REPO = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts/controls_kaggle_setup.py').is_file()), None)
    if REPO is None:
        spec = importlib.util.find_spec('scripts.controls_kaggle_setup')
        if spec is None:
            raise RuntimeError('Set CTRL_KAGGLE_REPO_ROOT to the intended checkout')
        REPO = Path(spec.origin).resolve().parents[1]
else:
    REPO = Path('/kaggle/working/glaucoma-thesis')
if not SMOKE and not Path('/kaggle').is_dir():
    raise RuntimeError('Real execution requires Kaggle; use CPU smoke locally')
if not REPO.exists() and not SMOKE and not os.environ.get('CTRL_KAGGLE_REPO_ROOT'):
    subprocess.run(['git', 'clone', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(REPO)], check=True)
SETUP_PATH = REPO / 'scripts/controls_kaggle_setup.py'
if not SETUP_PATH.is_file():
    raise RuntimeError(f'Stale/incomplete checkout: missing {SETUP_PATH}. Attach intended source; no automatic pull.')
if 'scripts.controls_kaggle_setup' in sys.modules:
    ks = sys.modules['scripts.controls_kaggle_setup']
else:
    spec = importlib.util.spec_from_file_location('scripts.controls_kaggle_setup', SETUP_PATH)
    ks = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = ks
    spec.loader.exec_module(ks)
if ks.BOOTSTRAP_API_VERSION != 1 or not all(hasattr(ks, name) for name in ('begin_setup', 'complete_setup', 'bootstrap', 'require_ready', 'validate_repo', 'configure_caches', 'record_imports')):
    raise RuntimeError('Incompatible bootstrap API; attach matching source and restart kernel')
ks.begin_setup(smoke=SMOKE)
ks.validate_repo(REPO, display=not SMOKE)
ks.configure_caches({'temp_root': os.environ.get('CTRL_KAGGLE_TEMP_ROOT', '/kaggle/temp/controls96'), 'output_root': os.environ.get('CTRL_KAGGLE_OUTPUT_ROOT', tempfile.gettempdir() if SMOKE else '/kaggle/working/controls96')})
BOOTSTRAP = ks.bootstrap(smoke=SMOKE, auto_repair_p100=AUTO_REPAIR_P100, defer_ready=True)
sys.path.insert(0, str(REPO))
from scripts import controls_kaggle as kg
from scripts import controls_kaggle_data as kd
ks.record_imports()
ks.validate_repo(REPO, display=False)
kg.load_credentials(smoke=SMOKE)
print('[setup]', BOOTSTRAP['status'], '| imports verified; no parent CUDA initialization')
ks.complete_setup(smoke=SMOKE)

## 2. Grouped Configuration
Only edit runtime, locations, and recovery settings. `RUN_TARGET` optionally selects one exact job such as `B4_s42`. `MAX_RUNS_PER_SESSION=None` removes the count cap, not the time/disk limits. Resume root points to the previous **group directory**, not a raw data directory. Artifact refs map a tag to `entity/glaucoma-thesis/controls-RUN_ID:latest`; `_summary` can point to the group summary artifact.

In [ ]:
CFG = kg.default_config()
# ===== RUN IDENTITY & PERSISTENCE (edit group only for a new experiment; set recovery locations) =====
RUN_GROUP = CFG['group']
RUN_TARGET = CFG['run_target']
RESUME_ROOT = CFG['resume_root']
ARTIFACT_REFS = {}
# ===== RUNTIME (edit session subset and hardware scheduling, never study budget) =====
GPU_MODE = 'auto'
MAX_RUNS_PER_SESSION = None if SMOKE else 2
SESSION_HOURS = 10.0
VRAM_FRACTION = 0.80
# ===== DATA LOCATIONS (set attached read-only roots or leave empty for selective HF downloads) =====
RAW_ROOT = CFG['raw_root']
BILATERAL_ROOT = CFG['bilateral_root']
BM3D_ROOT = CFG['bm3d_root']
# ===== FROZEN STUDY (do not edit; P, B3 seed 44 only, C1, C2, B4, BM; B1/B2 disabled) =====
SPECS = kg.SPECS
STORE_RES, MODEL_RES3D, RES2D = CFG['store_res'], CFG['res3d'], CFG['res2d']
EPOCHS, PATIENCE, SEEDS = CFG['epochs'], CFG['patience'], CFG['seeds']
LR, WD, EFFECTIVE_BATCH = 1e-4, 1e-4, 16
RUN_XAI = RUN_INFO = False
# ===== DERIVED CONFIG (do not edit; outputs working, downloads/views temp, never attached inputs) =====
CFG.update(group=RUN_GROUP, run_target=RUN_TARGET, resume_root=RESUME_ROOT, artifact_refs=ARTIFACT_REFS, gpu_mode=GPU_MODE, max_runs_per_session=MAX_RUNS_PER_SESSION, session_hours=SESSION_HOURS, vram_fraction=VRAM_FRACTION, raw_root=RAW_ROOT, bilateral_root=BILATERAL_ROOT, bm3d_root=BM3D_ROOT)
kg.validate_config(CFG)
SELECTED = kg.jobs(CFG)
DEVICES = [] if SMOKE else kg.detect_gpus(GPU_MODE)
print('[config]', len(SELECTED), 'study jobs; session cap', MAX_RUNS_PER_SESSION, '| GPUs:', DEVICES)

### Recovery Authorization and Completion Inventory
Requested restores fail closed on missing/incomplete files or wrong identity; they never silently start fresh. A single `RUN_TARGET` with `RESUME_ROOT` is an explicit restore request. For a full queue, the verified prior group inventory distinguishes existing runs from new jobs. Cloud refs are inspected via small recovery/identity/completion manifests **before** applying the two-unfinished-job cap; no big checkpoints are downloaded merely to discover completion. Old metrics-only summaries are rejected.
Keep initialized recovery disabled normally. Opting in permits only a matching, explicitly logged identity-stage artifact with no `last.pt`, and starts that recorded run from its initial seed; this is not checkpoint continuation. Missing last state without this stage metadata is always an error.

In [ ]:
# ===== INITIALIZED RECOVERY (edit only to authorize restart of a verified identity-stage run) =====
ALLOW_INITIALIZED_RECOVERY = False
CFG['allow_initialized_recovery'] = ALLOW_INITIALIZED_RECOVERY
kg.validate_config(CFG)
kg.configure_runtime(CFG)
SESSION_SELECTED, SESSION_DEADLINE = kg.session_jobs(CFG)
print('[config] Verified unfinished session assignment:', [job[2] for job in SESSION_SELECTED])

## 3. D1: Pinned Raw and BM3D Sources
Raw revision `939a38876b7b9313162842ef2d44b7edc2b57020`. This cell prepares only the sources required by `SESSION_SELECTED` (`raw`, `bilateral`, `bm3d`). BM3D shards are assembled from `tqhuyen/harvard-gf-denoise-benchmark-v2` under `classical/bm3d/3375a321513938835d2c`, verified against LFS SHA256 and labels, and their provenance is checked here. Authenticated HF metadata/checksums are verified even for attached inputs, which are never modified. Downloads and hash sidecars go under `/kaggle/temp`; actual free space is checked before download.

In [ ]:
ks.require_ready(smoke=SMOKE)
NEEDED_KINDS = {spec.get('dataset', 'raw') for spec, _, _ in SESSION_SELECTED}
if 'bilateral' in NEEDED_KINDS:
    NEEDED_KINDS.add('raw')
PREPARED = {kind: kd.prepare_source(CFG, kind) for kind in ('raw', 'bilateral', 'bm3d') if kind in NEEDED_KINDS}
print('[data] D1 sources prepared:', sorted(PREPARED))

## 4. D2: Verify Existing Bilateral-200 Export
Only B4 uses bilateral, so this step runs only when `bilateral` is in `PREPARED`. No denoising/background process is started, changed, or stopped. Original bilateral revision `47632c96b206707fd6423ee5b4da159069f63eaf`. Stored bilateral-96 is forbidden: augmentation-before-resize and quantization differ. B4 trains from scratch with ImageNet 2D initialization, not warm-start weights. BM3D provenance is verified inside D1 `prepare_source`.

In [ ]:
ks.require_ready(smoke=SMOKE)
if 'bilateral' in PREPARED:
    kd.verify_bilateral(PREPARED['raw'], PREPARED['bilateral'], smoke=SMOKE)
print('[data] D2 bilateral verification complete (if selected); no denoise computation')

## 5. D3: Parent-Only Writable View Caches
Build once before workers. Source-200 projections: index 0 `slab_mip` (half=16), index 1 `aip_full`. These are heuristic projections, **not RNFL segmentation**. Hashes, projection/resize parameters, code and package versions must match before reuse. All source/views/dz/label hashes are in run identity, without absolute runtime paths.

In [ ]:
ks.require_ready(smoke=SMOKE)
for kind in PREPARED:
    PREPARED[kind] = kd.prepare_views(CFG, PREPARED[kind])
print('[views] Parent preparation complete; workers only read caches')

## 6. D4: Dataset Factory
Reuse frozen paired flips/rotation/intensity augmentation seeded by (seed, epoch, index), then resize 3D to 96. B3 does not open or read 3D volumes. Dataset construction never invokes the original adjacent-source cache builder.

In [ ]:
ks.require_ready(smoke=SMOKE)
def make_datasets(spec, seed):
    return kd.make_datasets(PREPARED, spec, seed, smoke=SMOKE)
print('[dataset] Read-only dataset factory ready')

## 7. T1: Per-Spec Model Factory
`cm.ControlsModel` is reused unchanged. Restore last checkpoint first; new runs alone download ImageNet weights. Each process sees only logical `cuda:0`, selected via `CUDA_VISIBLE_DEVICES` before interpreter/Torch startup.

In [ ]:
ks.require_ready(smoke=SMOKE)
def make_model(spec, resume=False):
    return kg.make_model(spec, smoke=SMOKE, resume=resume)
print('[model] Model factory ready; no model allocated by parent')

## 8. T2: Probe and Trainer
Per-device 80% VRAM budget (~12.8 GiB on a 16 GiB GPU), candidates 16/8/4/2/1, effective batch fixed at 16. FP16 only, no BF16/TF32. Probe permits eight overflow skips but requires two successful updates; restores model/buffers/modes/RNG and discards its optimizer. Saved batch/scaler are reused on resume. Fresh trainer starts at the settled probe scale. Local atomic checkpoint every 10 steps; W&B full-state checkpoint each epoch and on stop.
The child initializes a separate `runtime-probe` W&B run before constructing/downloading a model or probing. The scientific training run has its own stable identity after the batch is resolved; runtime runs are excluded from completed-study summaries.

In [ ]:
ks.require_ready(smoke=SMOKE)
Trainer = kg.KaggleTrainer
probe_batch = kg.probe_batch
print('[batch] Probe runs inside assigned worker only, never in parent')

## 9. E1/E2: Evaluation Callback Definitions
Train full metrics per optimizer step; validation and test every epoch. Test is monitoring only; best checkpoint selected by validation AUC. The worker invokes these callbacks and publishes checkpoints after local atomic writes. Upload failures stop the session with local files retained.

In [ ]:
ks.require_ready(smoke=SMOKE)
def val_callback(dataset, batch_size):
    return kg.evaluate_callback(dataset, batch_size, CFG['num_workers'], 'val')
def test_callback(dataset, batch_size):
    return kg.evaluate_callback(dataset, batch_size, CFG['num_workers'], 'test')
print('[eval] E1/E2 callbacks defined; no training or inference yet')

## 10. T3: Dispatch/Fit (Training Happens Only Here)
Whole queue is 16 jobs; session cap defaults to 2. Interrupt writes a stop marker, stops dispatch, and waits up to 600 seconds for owned workers to checkpoint safely. It never kills unrelated jobs. If grace expires, children remain explicitly tracked in `kg.OWNED_CHILDREN`; do not close the kernel and call `kg.wait_owned(STOP_PATH, 600)` again. A cell returning does not prove Kaggle has persisted outputs.

In [ ]:
ks.require_ready(smoke=SMOKE)
STOP_PATH = Path(CFG['output_root']) / RUN_GROUP / 'STOP.json'
DISPATCHED = kg.dispatch(CFG, PREPARED)
print('[train] Dispatched session subset:', DISPATCHED)

## 11. E3/E4: Best Calibrated Reports and Independent Rerender
Workers persist best weights and raw train/val/test logits once, using `ct.calibrated_report`: validation temperature + Youden threshold, full calibrated metrics + bootstrap CI. Frozen AP/ECE semantics are retained, not silently corrected. This cell rerenders CSV/JSON from saved logits only: no GPU, model download, or retraining. No metric PNGs.
Replay validates the best-weight SHA256, scientific identity, source/view/label identities, split sizes, label digests and atomic NPZ receipt before recalibration. Stale or partial data is rejected. Each replay logs split tables to a linked `report-replay` W&B run and publishes acknowledged report artifacts; offline mode is permitted only for synthetic smoke.

In [ ]:
ks.require_ready(smoke=SMOKE)
kg.rerender_reports(CFG)
print('[report] E3/E4 tables available under', Path(CFG['output_root']) / RUN_GROUP)

## 12. E5/E6: Disabled by Authorized Scope
X-AI and information-theory analyses are intentionally disabled to preserve the controls budget. No figures, additional training, or GPU analysis is executed.

In [ ]:
ks.require_ready(smoke=SMOKE)
assert not RUN_XAI and not RUN_INFO
print('[xai] E5 disabled; [info] E6 disabled')

## 13. Aggregate Available Seeds and Persist
Include persisted prior-session results, not just this session. Report mean, sample std and n_seeds; std is null with fewer than two observations, never fabricated as zero. Fewer than three seeds is marked partial. W&B artifact upload must acknowledge success. **Save Version or download outputs yourself before ending an interactive Kaggle session.** W&B versions consume storage: review quotas and manually manage retention; this adapter never deletes remote versions.

In [ ]:
ks.require_ready(smoke=SMOKE)
SUMMARY = kg.aggregate(CFG)
kg.persist_summary(CFG, SUMMARY)
print('[report] Completed runs:', len(SUMMARY['runs']), '/ 16; Save Version/download required')